# Qwen3-VL-4B-Instruct Benchmark on AWS Neuron

This notebook benchmarks [Qwen3-VL-4B-Instruct](https://huggingface.co/Qwen/Qwen3-VL-4B-Instruct) on AWS Neuron hardware (Trainium2 / Inferentia2) using NxD Inference (NxDI) via the vLLM interface.

**Supported instances**: `trn2.3xlarge` (LNC=2 or LNC=1), `inf2.8xlarge` (SDK 2.28 only)

**SDK**: AWS Neuron SDK 2.28 or 2.29
- **SDK 2.28**: `Deep Learning AMI Neuron (Ubuntu 24.04) 20260227`
- **SDK 2.29**: `Deep Learning AMI Neuron (Ubuntu 24.04) 20260410`

**SDK 2.29 notes**: NxD Inference 0.9 drops inf2/trn1 support (trn2 only). vLLM 0.16.0 + vllm-neuron 0.5.0. NKI 0.3.0 (GA).

**Key findings from benchmarking**:
- **tp=2 DP=2** gives highest aggregate throughput: 103.3 tok/s image+text, 116.7 tok/s text-only
- **tp=4 DP=1** gives lowest per-request latency: 91.8 tok/s image+text, 2.8s per request
- **MLP kernel hurts** at this model scale (-15.6% throughput at tp=4)
- **tp=1 is NOT viable** on SDK 2.28 (compiler bugs)
- **ISA kernels are REQUIRED** at tp=2 (without them, compiler ICE)

## Prerequisites

```bash
# Activate the pre-installed PyTorch Inference environment
# SDK 2.28:
source /opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/bin/activate
# SDK 2.29:
source /opt/aws_neuronx_venv_pytorch_inference_vllm_0_16/bin/activate

# Verify Neuron devices
neuron-ls
```

## Step 1: Configuration

Edit the cell below to configure the benchmark. The notebook auto-detects your instance type and sets safe defaults.

In [ ]:
import subprocess
import os

# ============================================================
# AUTO-DETECTION (you can override below)
# ============================================================
result = subprocess.run(['neuron-ls'], capture_output=True, text=True)
output = result.stdout

# Parse instance type
instance_type = 'unknown'
if 'instance-type:' in output:
    instance_type = output.split('instance-type:')[1].split('\n')[0].strip()

# Parse core count from the table data rows (format: | 0      | 4      | 0-3      | ...)
num_cores = 0
for line in output.split('\n'):
    parts = [p.strip() for p in line.split('|') if p.strip()]
    if len(parts) >= 3 and parts[0].isdigit() and parts[1].isdigit():
        num_cores += int(parts[1])  # Sum cores across all devices

if num_cores == 0:
    # Fallback: count rows with digit-starting device IDs
    print(f"WARNING: Could not parse core count from neuron-ls. Output:\n{output}")
    num_cores = 2  # Safe default

# Parse LNC config
lnc_config = 2  # default
if 'logical-neuroncore-config:' in output:
    lnc_config = int(output.split('logical-neuroncore-config:')[1].split('\n')[0].strip())

if 'inf2' in instance_type:
    DETECTED_PLATFORM = 'inf2'
    DETECTED_CORES = num_cores
elif lnc_config == 1 or num_cores > 4:
    DETECTED_PLATFORM = 'trn2_lnc1'
    DETECTED_CORES = num_cores
else:
    DETECTED_PLATFORM = 'trn2_lnc2'
    DETECTED_CORES = num_cores

print(f"Detected: {instance_type}, {num_cores} cores, LNC={lnc_config} -> platform={DETECTED_PLATFORM}")

# ============================================================
# USER CONFIGURATION - Edit these values
# ============================================================

# Model
MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"  # HuggingFace model ID
MODEL_PATH = os.path.expanduser("~/models/Qwen3-VL-4B-Instruct")  # Local download path (also check /mnt/models/)
if os.path.exists("/mnt/models/Qwen3-VL-4B-Instruct"):
    MODEL_PATH = "/mnt/models/Qwen3-VL-4B-Instruct"  # Use existing model on NVMe

# Parallelism
TP_DEGREE = 2       # Tensor parallel degree: 2 or 4 (tp=1 NOT supported on SDK 2.28)
                     # tp=2: 61.7 tok/s single-req, enables DP=2
                     # tp=4: 95.1 tok/s single-req, uses all cores on LNC=2

# Sequence length
SEQ_LEN = 4096      # Max sequence length: 1024, 2048, 4096, or 8192

# ISA Kernels
QKV_KERNEL_ENABLED = True   # QKV fused kernel (REQUIRED at tp=2, recommended at tp=4)
ATTN_KERNEL_ENABLED = True  # Attention kernel (REQUIRED at tp=2, recommended at tp=4)
MLP_KERNEL_ENABLED = False  # MLP kernel (only works at tp>=4, HURTS performance at this scale)

# Generation
MAX_NEW_TOKENS = 256  # Max tokens to generate per request

# Benchmark
WARMUP_RUNS = 2       # Number of warmup iterations (discarded)
BENCHMARK_RUNS = 5    # Number of timed iterations

# LNC setting (trn2 only)
LNC_MODE = 2  # 1 or 2 (must match /etc/environment NEURON_LOGICAL_NC_CONFIG)

In [ ]:
# ============================================================
# VALIDATION - Checks for incompatible configurations
# ============================================================

errors = []
warnings = []

# tp=1 is not supported
if TP_DEGREE == 1:
    errors.append(
        "tp=1 is NOT supported on SDK 2.28 for Qwen3-VL-4B. "
        "QKV kernel I=6144 > 4096 ISA limit (with kernels), "
        "compiler ICE NCC_IBIR182 (without kernels). Use tp=2 or tp=4."
    )

# tp must not exceed available cores
if TP_DEGREE > DETECTED_CORES:
    errors.append(
        f"tp={TP_DEGREE} exceeds available cores ({DETECTED_CORES}). "
        f"Max TP for {DETECTED_PLATFORM} is {DETECTED_CORES}."
    )

# inf2: ISA kernels must be off
if DETECTED_PLATFORM == 'inf2':
    # SDK 2.29 drops inf2 support in NxDI 0.9
    try:
        import neuronx_distributed_inference
        nxdi_ver = getattr(neuronx_distributed_inference, '__version__', '0.0')
        if nxdi_ver.startswith('0.9') or nxdi_ver.startswith('0.1'):
            errors.append(
                f"NxD Inference {nxdi_ver} does NOT support inf2/trn1 (trn2 only). "
                "Use SDK 2.28 DLAMI for inf2 instances."
            )
    except ImportError:
        pass
    if QKV_KERNEL_ENABLED or ATTN_KERNEL_ENABLED or MLP_KERNEL_ENABLED:
        warnings.append(
            "ISA kernels are NOT supported on inf2 (compiler bug NCC_IBVF032). Disabling all kernels."
        )
        QKV_KERNEL_ENABLED = False
        ATTN_KERNEL_ENABLED = False
        MLP_KERNEL_ENABLED = False
    # inf2 + tp=2 + no kernels may trigger NCC_ITEN404 (needs testing)
    warnings.append(
        "inf2 with kernels off may trigger compiler ICE NCC_ITEN404. "
        "This has been observed on trn2 but may not reproduce on inf2."
    )
    # inf2 only supports LNC=1
    if LNC_MODE != 1:
        warnings.append(
            f"LNC_MODE={LNC_MODE} not supported on inf2 (NeuronCore v2 only supports LNC=1). Setting LNC_MODE=1."
        )
        LNC_MODE = 1

# trn2 tp=2: kernels are REQUIRED
if DETECTED_PLATFORM.startswith('trn2') and TP_DEGREE == 2:
    if not QKV_KERNEL_ENABLED or not ATTN_KERNEL_ENABLED:
        warnings.append(
            "QKV and Attention kernels are REQUIRED at tp=2 on trn2 (compiler ICE NCC_ITEN404 without them). Enabling."
        )
        QKV_KERNEL_ENABLED = True
        ATTN_KERNEL_ENABLED = True

# MLP kernel size check
INTERMEDIATE_SIZE = 9728
if MLP_KERNEL_ENABLED and INTERMEDIATE_SIZE // TP_DEGREE > 4096:
    warnings.append(
        f"MLP kernel requires intermediate_size/tp <= 4096. "
        f"{INTERMEDIATE_SIZE}/{TP_DEGREE} = {INTERMEDIATE_SIZE // TP_DEGREE} > 4096. Disabling MLP kernel."
    )
    MLP_KERNEL_ENABLED = False

# MLP kernel performance warning at tp=4
if MLP_KERNEL_ENABLED and TP_DEGREE == 4:
    warnings.append(
        "MLP kernel HURTS performance at tp=4 for this model (-15.6% throughput). "
        "Consider setting MLP_KERNEL_ENABLED = False."
    )

# LNC mode check
if DETECTED_PLATFORM.startswith('trn2'):
    expected_cores_lnc2 = 4  # trn2.3xlarge
    expected_cores_lnc1 = 8
    if LNC_MODE == 2 and DETECTED_CORES != expected_cores_lnc2:
        warnings.append(f"LNC_MODE=2 but detected {DETECTED_CORES} cores (expected {expected_cores_lnc2}). Check NEURON_LOGICAL_NC_CONFIG.")
    if LNC_MODE == 1 and DETECTED_CORES != expected_cores_lnc1:
        warnings.append(f"LNC_MODE=1 but detected {DETECTED_CORES} cores (expected {expected_cores_lnc1}). Check NEURON_LOGICAL_NC_CONFIG.")

# Print results
for e in errors:
    print(f"ERROR: {e}")
for w in warnings:
    print(f"WARNING: {w}")

if errors:
    raise ValueError("Fix the above errors before proceeding.")

print(f"\nConfiguration validated:")
print(f"  Platform:    {DETECTED_PLATFORM} ({DETECTED_CORES} cores)")
print(f"  TP degree:   {TP_DEGREE}")
print(f"  Seq length:  {SEQ_LEN}")
print(f"  QKV kernel:  {QKV_KERNEL_ENABLED}")
print(f"  Attn kernel: {ATTN_KERNEL_ENABLED}")
print(f"  MLP kernel:  {MLP_KERNEL_ENABLED}")
print(f"  Max tokens:  {MAX_NEW_TOKENS}")

## Step 2: Install Dependencies and Download Model

In [ ]:
!pip install -q qwen-vl-utils pillow

In [ ]:
import time

if not os.path.exists(MODEL_PATH):
    print(f"Downloading {MODEL_ID} to {MODEL_PATH}...")
    from huggingface_hub import snapshot_download
    t0 = time.time()
    snapshot_download(MODEL_ID, local_dir=MODEL_PATH)
    print(f"Download complete in {time.time() - t0:.1f}s")
else:
    print(f"Model already exists at {MODEL_PATH}")

## Step 3: Build Neuron Configuration

The configuration is auto-generated from your settings above. Two separate configs are needed:
- **text_neuron_config**: For the autoregressive text model (context encoding + token generation)
- **vision_neuron_config**: For the vision encoder (no ISA kernels, different bucket structure)

In [ ]:
# Build bucket lists based on SEQ_LEN
def make_buckets(seq_len):
    """Generate appropriate bucket list for the given seq_len."""
    candidates = [512, 1024, 2048, 4096, 8192]
    return [b for b in candidates if b <= seq_len]

text_buckets = make_buckets(SEQ_LEN)
vision_buckets = [b for b in [1024, 4096] if b <= SEQ_LEN]

text_neuron_config = {
    "batch_size": 1,
    "ctx_batch_size": 1,
    "tkg_batch_size": 1,
    "seq_len": SEQ_LEN,
    "max_context_length": SEQ_LEN,
    "torch_dtype": "bfloat16",
    "tp_degree": TP_DEGREE,
    "world_size": TP_DEGREE,
    "enable_bucketing": True,
    "context_encoding_buckets": text_buckets,
    "token_generation_buckets": text_buckets,
    "fused_qkv": True,
    "qkv_kernel_enabled": QKV_KERNEL_ENABLED,
    "mlp_kernel_enabled": MLP_KERNEL_ENABLED,
    "attn_kernel_enabled": ATTN_KERNEL_ENABLED,
    "logical_neuron_cores": LNC_MODE,
    "cc_pipeline_tiling_factor": LNC_MODE,
    "rpl_reduce_dtype": "bfloat16",
    "attention_dtype": "bfloat16",
    "cast_type": "as-declared",
}

vision_neuron_config = {
    "batch_size": 1,
    "seq_len": min(SEQ_LEN, 4096),  # Vision encoder caps at 4096
    "max_context_length": min(SEQ_LEN, 4096),
    "enable_bucketing": True,
    "buckets": vision_buckets,
    "world_size": TP_DEGREE,
    "tp_degree": TP_DEGREE,
    "torch_dtype": "bfloat16",
    "rpl_reduce_dtype": "bfloat16",
    "cast_type": "as-declared",
    "logical_neuron_cores": LNC_MODE,
    "cc_pipeline_tiling_factor": LNC_MODE,
    "fused_qkv": True,
    "attn_kernel_enabled": False,  # Vision encoder: ISA kernels not supported
    "mlp_kernel_enabled": False,
}

print("Text neuron config:")
for k, v in text_neuron_config.items():
    print(f"  {k}: {v}")
print(f"\nVision neuron config:")
for k, v in vision_neuron_config.items():
    print(f"  {k}: {v}")

## Step 4: Apply Patches and Compile Model

The 4B model has `tie_word_embeddings=true`, which crashes NxDI during weight loading.
NxDI's `NeuronQwen3VLForCausalLM` is missing the `update_state_dict_for_tied_weights` method
(the text submodel has it, but the top-level VLM class does not).

**SDK 2.28**: In-process monkey-patch works since vLLM 0.13 runs model loading in the same process.

**SDK 2.29**: vLLM 0.16 spawns a separate `EngineCore_DP0` process via `multiprocessing.spawn`.
Monkey-patches applied in the main process do NOT propagate to the spawned process.
We must patch the installed source file on disk so the spawned process imports the fixed version.

**Note**: First compilation takes 2-8 minutes depending on SDK version and instance. Subsequent runs use cached NEFFs (~30-90s).

In [ ]:
import importlib
import inspect

# Detect SDK version to choose the right patching strategy
try:
    import vllm_neuron
    vllm_neuron_ver = getattr(vllm_neuron, '__version__', 'unknown')
except ImportError:
    vllm_neuron_ver = 'not installed'

from neuronx_distributed_inference.models.qwen3_vl.modeling_qwen3_vl import (
    NeuronQwen3VLForCausalLM,
)

# Check if the class already has the method (future SDK fix)
has_own = 'update_state_dict_for_tied_weights' in NeuronQwen3VLForCausalLM.__dict__
if has_own:
    src = inspect.getsource(NeuronQwen3VLForCausalLM.update_state_dict_for_tied_weights)
    if 'NotImplementedError' not in src:
        print(f'NeuronQwen3VLForCausalLM already has update_state_dict_for_tied_weights. No patch needed.')
    else:
        has_own = False  # Base class stub, still needs patching

if not has_own:
    if vllm_neuron_ver.startswith('0.5') or vllm_neuron_ver.startswith('0.6'):
        # SDK 2.29+: vLLM 0.16 uses multiprocessing.spawn — must patch file on disk
        print(f'SDK 2.29 detected (vllm-neuron {vllm_neuron_ver}). Patching source file on disk...')
        src_file = inspect.getfile(NeuronQwen3VLForCausalLM)
        with open(src_file, 'r') as f:
            content = f.read()
        
        patch_marker = '# PATCHED: update_state_dict_for_tied_weights'
        if patch_marker not in content:
            old = '    def __init__(self, *args, **kwargs):\n        super().__init__(\n            self.text_model_cls,'
            new = ('    @staticmethod  ' + patch_marker + '\n'
                   '    def update_state_dict_for_tied_weights(state_dict):\n'
                   '        state_dict["lm_head.weight"] = state_dict["embed_tokens.weight"].clone()\n'
                   '\n'
                   '    def __init__(self, *args, **kwargs):\n'
                   '        super().__init__(\n'
                   '            self.text_model_cls,')
            assert old in content, f'Could not find patch target in {src_file}'
            content = content.replace(old, new, 1)
            with open(src_file, 'w') as f:
                f.write(content)
            # Clear bytecode cache
            import pathlib, shutil
            for pyc in pathlib.Path(src_file).parent.rglob('*.pyc'):
                pyc.unlink()
            for cache_dir in pathlib.Path(src_file).parent.rglob('__pycache__'):
                shutil.rmtree(cache_dir, ignore_errors=True)
            # Reload to verify
            import neuronx_distributed_inference.models.qwen3_vl.modeling_qwen3_vl as mod
            importlib.reload(mod)
            from neuronx_distributed_inference.models.qwen3_vl.modeling_qwen3_vl import NeuronQwen3VLForCausalLM
            assert 'update_state_dict_for_tied_weights' in NeuronQwen3VLForCausalLM.__dict__
            print(f'Patched {src_file} on disk. Spawned process will import the fix.')
        else:
            print('File already patched from a previous run.')
    else:
        # SDK 2.28: in-process monkey-patch is sufficient
        print(f'SDK 2.28 detected (vllm-neuron {vllm_neuron_ver}). Applying in-process monkey-patch...')
        @staticmethod
        def update_state_dict_for_tied_weights(state_dict):
            if 'lm_head.weight' not in state_dict and 'embed_tokens.weight' in state_dict:
                print('[PATCH] tie_word_embeddings: copying embed_tokens -> lm_head')
                state_dict['lm_head.weight'] = state_dict['embed_tokens.weight'].clone()
        NeuronQwen3VLForCausalLM.update_state_dict_for_tied_weights = update_state_dict_for_tied_weights
        print('Patched in-process: tie_word_embeddings handler for 4B model')

In [ ]:
os.environ["VLLM_NEURON_FRAMEWORK"] = "neuronx-distributed-inference"

from vllm import LLM, SamplingParams

print(f"Compiling model with tp={TP_DEGREE}, seq_len={SEQ_LEN}...")
print(f"Kernels: QKV={QKV_KERNEL_ENABLED}, Attn={ATTN_KERNEL_ENABLED}, MLP={MLP_KERNEL_ENABLED}")

compile_start = time.time()
llm = LLM(
    model=MODEL_PATH,
    tokenizer=MODEL_PATH,
    trust_remote_code=True,
    dtype="bfloat16",
    tensor_parallel_size=TP_DEGREE,
    max_num_seqs=1,
    max_model_len=SEQ_LEN,
    additional_config=dict(
        override_neuron_config=dict(
            text_neuron_config=text_neuron_config,
            vision_neuron_config=vision_neuron_config,
        )
    ),
    limit_mm_per_prompt={"image": 1},
    enable_prefix_caching=False,
    enable_chunked_prefill=False,
)
compile_time = time.time() - compile_start
print(f"\nModel ready in {compile_time:.1f}s")

## Step 5: Prepare Sample Images

Images are resized to stay within the vision encoder's patch budget (max 4096 visual tokens).
Each 56×56 pixel block becomes one visual token after the spatial merge.

In [ ]:
import urllib.request
from io import BytesIO
from PIL import Image

MAX_LONG_SIDE = 1024  # Keep images within 4096-token vision budget

def load_and_resize(source, max_long_side=MAX_LONG_SIDE):
    """Load image from URL or path and resize to fit within patch budget."""
    if source.startswith('http'):
        req = urllib.request.Request(source, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=30) as resp:
            img = Image.open(BytesIO(resp.read())).convert("RGB")
    else:
        img = Image.open(source).convert("RGB")

    w, h = img.size
    if max(w, h) > max_long_side:
        scale = max_long_side / max(w, h)
        new_w = int(w * scale) // 56 * 56  # Align to patch grid (56 = patch_size * spatial_merge)
        new_h = int(h * scale) // 56 * 56
        new_w = max(new_w, 56)
        new_h = max(new_h, 56)
        img = img.resize((new_w, new_h), Image.LANCZOS)
        print(f"  Resized: {w}x{h} -> {new_w}x{new_h} (~{new_w * new_h // 56 // 56} visual tokens)")
    else:
        print(f"  Size: {w}x{h} (~{w * h // 56 // 56} visual tokens)")
    return img

# Load sample images
print("Loading sample images...")
print("Image 1 (Qwen demo - woman with dog on beach):")
img1 = load_and_resize("https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg")

print("Image 2 (landscape):")
try:
    img2 = load_and_resize("https://picsum.photos/id/24/800/600")
except Exception as e:
    print(f"  Failed to load ({e}), using synthetic image")
    img2 = Image.new("RGB", (672, 504), color=(100, 150, 200))

print("Done.")

## Step 6: Inference Helper

In [ ]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)

def run_inference(llm, prompt_text, image=None, max_tokens=MAX_NEW_TOKENS):
    """Run inference and return result dict with timing."""
    if image is not None:
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt_text},
            ],
        }]
    else:
        messages = [{"role": "user", "content": prompt_text}]

    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    if image is not None:
        inputs = {"prompt": prompt, "multi_modal_data": {"image": [image]}}
    else:
        inputs = {"prompt": prompt}

    sampling_params = SamplingParams(top_k=1, max_tokens=max_tokens, temperature=0.0)

    t0 = time.time()
    outputs = llm.generate([inputs], sampling_params=sampling_params)
    latency = time.time() - t0

    text = outputs[0].outputs[0].text
    num_tokens = len(outputs[0].outputs[0].token_ids)
    tok_per_sec = num_tokens / latency if latency > 0 else 0

    return {
        "text": text,
        "num_tokens": num_tokens,
        "latency_s": latency,
        "tok_per_sec": tok_per_sec,
    }

print("Inference helper ready.")

## Step 7: Example — Text-Only Inference

In [ ]:
print("=" * 60)
print("TEXT-ONLY INFERENCE")
print("=" * 60)

text_prompt = "Explain what a vision-language model is and how it works. Be concise."
print(f"Prompt: {text_prompt}\n")

result = run_inference(llm, text_prompt)
print(f"Response ({result['num_tokens']} tokens, {result['latency_s']:.2f}s, {result['tok_per_sec']:.1f} tok/s):")
print(result['text'][:2000])

## Step 8: Example — Image + Text Inference

In [ ]:
print("=" * 60)
print("IMAGE + TEXT INFERENCE")
print("=" * 60)

# Image description
img_prompt = "Describe this image in detail. What do you see?"
print(f"\nImage 1: Qwen demo (woman with dog)")
print(f"Prompt: {img_prompt}\n")

result = run_inference(llm, img_prompt, image=img1)
print(f"Response ({result['num_tokens']} tokens, {result['latency_s']:.2f}s, {result['tok_per_sec']:.1f} tok/s):")
print(result['text'][:2000])

In [ ]:
# Visual Q&A
qa_prompt = "How many people are in this image? What are they doing?"
print(f"\nImage 2: Landscape")
print(f"Prompt: {qa_prompt}\n")

result = run_inference(llm, qa_prompt, image=img2)
print(f"Response ({result['num_tokens']} tokens, {result['latency_s']:.2f}s, {result['tok_per_sec']:.1f} tok/s):")
print(result['text'][:2000])

## Step 9: Benchmark Suite

Runs timed iterations for both text-only and image+text prompts.
Reports mean, std, P50, P99 latency and throughput.

In [ ]:
import numpy as np

def run_benchmark(llm, prompt_text, image=None, warmup=WARMUP_RUNS, runs=BENCHMARK_RUNS, label=""):
    """Run warmup + timed benchmark iterations."""
    print(f"\n{'=' * 60}")
    print(f"BENCHMARK: {label}")
    print(f"  Warmup: {warmup} runs, Benchmark: {runs} runs")
    print(f"{'=' * 60}")

    # Warmup
    for i in range(warmup):
        r = run_inference(llm, prompt_text, image=image)
        print(f"  Warmup {i+1}: {r['latency_s']:.2f}s, {r['tok_per_sec']:.1f} tok/s, {r['num_tokens']} tokens")

    # Timed runs
    latencies = []
    throughputs = []
    token_counts = []

    for i in range(runs):
        r = run_inference(llm, prompt_text, image=image)
        latencies.append(r['latency_s'])
        throughputs.append(r['tok_per_sec'])
        token_counts.append(r['num_tokens'])
        print(f"  Run {i+1}: {r['latency_s']:.2f}s, {r['tok_per_sec']:.1f} tok/s, {r['num_tokens']} tokens")

    lat = np.array(latencies)
    tp = np.array(throughputs)
    toks = np.array(token_counts)

    results = {
        "label": label,
        "latency_mean": float(lat.mean()),
        "latency_std": float(lat.std()),
        "latency_p50": float(np.percentile(lat, 50)),
        "latency_p99": float(np.percentile(lat, 99)),
        "throughput_mean": float(tp.mean()),
        "throughput_std": float(tp.std()),
        "throughput_min": float(tp.min()),
        "throughput_max": float(tp.max()),
        "tokens_mean": float(toks.mean()),
    }

    print(f"\n  --- Results ---")
    print(f"  Throughput: {results['throughput_mean']:.1f} tok/s (std={results['throughput_std']:.1f}, min={results['throughput_min']:.1f}, max={results['throughput_max']:.1f})")
    print(f"  Latency:   {results['latency_mean']:.3f}s (std={results['latency_std']:.3f}, P50={results['latency_p50']:.3f}, P99={results['latency_p99']:.3f})")
    print(f"  Tokens:    {results['tokens_mean']:.0f} mean")

    return results

print("Benchmark function ready.")

In [ ]:
# Run text-only benchmark
text_benchmark_prompt = "Explain what a vision-language model is and how it works. Be concise."
text_results = run_benchmark(llm, text_benchmark_prompt, label="Text-Only")

In [ ]:
# Run image+text benchmark (using image 1 for consistency)
img_benchmark_prompt = "Describe this image in detail. What do you see?"
img_results = run_benchmark(llm, img_benchmark_prompt, image=img1, label="Image+Text")

## Step 10: Results Summary

In [ ]:
print("\n" + "=" * 70)
print("BENCHMARK RESULTS SUMMARY")
print("=" * 70)
print(f"\nConfiguration:")
print(f"  Instance:    {instance_type} ({DETECTED_PLATFORM})")
print(f"  TP degree:   {TP_DEGREE}")
print(f"  Seq length:  {SEQ_LEN}")
print(f"  LNC mode:    {LNC_MODE}")
print(f"  QKV kernel:  {QKV_KERNEL_ENABLED}")
print(f"  Attn kernel: {ATTN_KERNEL_ENABLED}")
print(f"  MLP kernel:  {MLP_KERNEL_ENABLED}")
print(f"  Max tokens:  {MAX_NEW_TOKENS}")
print(f"  Compile time: {compile_time:.1f}s")

print(f"\n{'Metric':<30} {'Text-Only':>12} {'Image+Text':>12}")
print("-" * 56)
print(f"{'Throughput (tok/s)':<30} {text_results['throughput_mean']:>11.1f}  {img_results['throughput_mean']:>11.1f}")
print(f"{'  std':<30} {text_results['throughput_std']:>11.1f}  {img_results['throughput_std']:>11.1f}")
print(f"{'  min':<30} {text_results['throughput_min']:>11.1f}  {img_results['throughput_min']:>11.1f}")
print(f"{'  max':<30} {text_results['throughput_max']:>11.1f}  {img_results['throughput_max']:>11.1f}")
print(f"{'Latency (s)':<30} {text_results['latency_mean']:>11.3f}  {img_results['latency_mean']:>11.3f}")
print(f"{'  P50':<30} {text_results['latency_p50']:>11.3f}  {img_results['latency_p50']:>11.3f}")
print(f"{'  P99':<30} {text_results['latency_p99']:>11.3f}  {img_results['latency_p99']:>11.3f}")
print(f"{'Avg tokens generated':<30} {text_results['tokens_mean']:>11.0f}  {img_results['tokens_mean']:>11.0f}")

In [ ]:
# Save results to JSON
import json

all_results = {
    "config": {
        "instance_type": instance_type,
        "platform": DETECTED_PLATFORM,
        "num_cores": DETECTED_CORES,
        "tp_degree": TP_DEGREE,
        "seq_len": SEQ_LEN,
        "lnc_mode": LNC_MODE,
        "qkv_kernel": QKV_KERNEL_ENABLED,
        "attn_kernel": ATTN_KERNEL_ENABLED,
        "mlp_kernel": MLP_KERNEL_ENABLED,
        "max_new_tokens": MAX_NEW_TOKENS,
        "compile_time_s": compile_time,
    },
    "text_only": text_results,
    "image_text": img_results,
}

results_file = f"results_{DETECTED_PLATFORM}_tp{TP_DEGREE}_seq{SEQ_LEN}.json"
with open(results_file, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"Results saved to {results_file}")

## Reference: Benchmark Results (trn2.3xlarge, SDK 2.28)

These results were collected on 2026-03-12 to 2026-03-16 with SDK 2.28, Qwen3-VL-4B-Instruct, `max_new_tokens=256`, `seq_len=4096`.

### NxDI Offline — Single Request Performance

| TP | LNC | Kernels | Image+Text tok/s | Text-Only tok/s | Compile (s) | Notes |
|----|-----|---------|-----------------|----------------|-------------|-------|
| 2 | 2 | QKV+Attn | 61.7 | 65.7 | 367 | LNC=2 baseline |
| 4 | 2 | QKV+Attn+MLP | 82.3 | 88.8 | 279 | MLP kernel works but hurts |
| **4** | **2** | **QKV+Attn** | **95.1** | **103.5** | 325 | **Fastest single-request** |
| 2 | 1 | QKV+Attn | 39.3 | 41.9 | 31 | LNC=1, -36% vs LNC=2 |
| 4 | 1 | QKV+Attn | 61.7 | 66.9 | 292 | LNC=1, -35% vs LNC=2 |
| **1** | **1** | **None** | **28.6** | **30.8** | 24 | **tp=1 works at LNC=1!** |
| 1 | 2 | Any | FAIL | FAIL | -- | QKV I=6144 > 4096 / ICE NCC_IBIR182 |
| 2 | 2 | None | FAIL | FAIL | -- | Compiler ICE NCC_ITEN404 |

### vLLM Serve — Aggregate Throughput

| TP | DP | LNC | Concurrent | Img+Text tok/s | Text-Only tok/s | Per-Req Latency (img) |
|----|-----|-----|-----------|---------------|----------------|---------------------|
| 4 | 1 | 2 | 1 | 91.8 | 103.1 | **2.79s (lowest)** |
| 2 | 1 | 2 | 1 | 59.8 | 65.5 | 4.28s |
| 2 | 2 | 2 | 2 | 103.3 | 116.7 | 4.94s |
| 2 | 2 | 1 | 2 | 70.9 | 83.4 | 7.20s |
| 4 | 2 | 1 | 2 | 110.7 | 133.5 | 4.60s |
| **2** | **4** | **1** | **4** | **134.3** | **162.1** | 7.55s |
| 1 | 4+ | 1 | -- | FAIL | FAIL | -- (OOM) |

### Concurrency Scaling (tp=2 DP=4 LNC=1)

| Concurrent | Img tok/s | Txt tok/s | Img latency | Errors |
|------------|-----------|-----------|-------------|--------|
| 1 | 37.6 | 41.6 | 6.81s | 0 |
| 2 | 72.0 | 83.3 | 7.08s | 0 |
| 3 | 104.6 | 125.6 | 7.32s | 0 |
| **4** | **134.3** | **162.1** | **7.55s** | **0** |
| 5+ | 0 | 0 | -- | ALL fail |

Scaling is near-perfect linear (~96% efficiency). Hard wall at concurrent > DP workers.

### Sequence Length Impact

| TP | seq_len | Img tok/s | Txt tok/s | vs seq=4096 |
|----|---------|-----------|-----------|-------------|
| 2 | 1024 | 39.2 | 41.7 | -0.2% / -0.5% |
| 2 | 4096 | 39.3 | 41.9 | baseline |
| 4 | 1024 | 62.0 | 67.2 | +0.5% / +0.5% |
| 4 | 4096 | 61.7 | 66.9 | baseline |

**Verdict**: Reducing max_model_len has <1% impact. KV cache is negligible vs model weights.

### Key Findings

1. **For maximum throughput (serving)**: Use **tp=2 DP=4 at LNC=1** via vLLM serve. 134 img tok/s, 162 text tok/s.

2. **For minimum latency (interactive)**: Use **tp=4 DP=1 at LNC=2**. 2.8s per image request. 95 tok/s single-request.

3. **LNC=1 wins for throughput, LNC=2 wins for latency**: Despite ~35% slower per-worker, LNC=1 enables 2x DP workers, yielding +29%/+43% aggregate throughput.

4. **Concurrency = DP**: With bs=1, each DP worker handles exactly one request. Throughput scales linearly up to DP, then fails.

5. **MLP kernel hurts at this scale**: -15.6% throughput at tp=4. ISA dispatch overhead > compute benefit.

6. **ISA kernels mandatory at tp=2**: Without them, compiler ICE NCC_ITEN404.

7. **tp=1 works at LNC=1 only**: All kernels off, 28.6 img tok/s. Blocked at LNC=2 by ICE.

8. **Seq length and -O2 optimization have zero effect**: Both tested and confirmed <1% impact.

9. **vLLM overhead is minimal**: ~3% for image+text, <1% for text-only vs NxDI offline.

10. **Batch > 1 is fundamentally blocked**: KV cache manager asserts `seq_ids.shape[0] == 1` in continuous batching. DP is the only concurrency path.


## Reference: Configuration Guide

### Which config should I use?

| Scenario | TP | DP | LNC | Framework | Expected Performance |
|----------|----|----|-----|-----------|---------------------|
| **Max throughput (serving)** | 2 | 4 | 1 | vLLM serve | ~134 img / ~162 text tok/s |
| **Balanced throughput + latency** | 4 | 2 | 1 | vLLM serve | ~111 img / ~134 text tok/s, 4.6s/req |
| **Min latency (interactive)** | 4 | 1 | 2 | vLLM serve or NxDI | ~92-95 img / ~103 text tok/s, 2.8s/req |
| **Quick testing** | 2 | 1 | 2 | NxDI offline | ~62 img / ~66 text tok/s |
| **Single-core experiment** | 1 | 1 | 1 | NxDI offline | ~29 img / ~31 text tok/s (all kernels off) |

### LNC Mode Selection

| LNC | Logical Cores | Best For | Key Tradeoff |
|-----|--------------|----------|-------------|
| **LNC=2** (default) | 4 | Latency-sensitive | Wider cores = faster per-request, but max DP=2 at tp=2 |
| **LNC=1** | 8 | Throughput-oriented | Narrower cores (-35%), but max DP=4 at tp=2 = more aggregate |



### Known SDK 2.28 Limitations for this Model

- **tie_word_embeddings=true** crashes weight loading (patched in this notebook)
- **tp=1** not viable at LNC=2 (QKV kernel limit + compiler ICE). Works at LNC=1 with all kernels off.
- **Kernels-off at tp=2** triggers compiler ICE NCC_ITEN404 (kernels route around bug)
- **MLP kernel** overhead exceeds benefit at this model scale (-15.6%)
- **batch_size > 1** blocked by KV cache assertion (SDK limitation)
- **Concurrent > DP** causes 500 errors (no request queuing with bs=1)
- **vLLM DP > 1** requires pre-compilation workaround (compile race condition)
- **Image resize** required client-side for vLLM serve (max ~1M pixels)
- **Bucket size < 512** triggers NCC_IBIR039 compiler ICE
- **-O2 compiler flag** shows no improvement over -O1
- **Reduced max_model_len** has <1% throughput impact

### SDK 2.29 Changes

- **NxD Inference 0.9** drops inf2/trn1 support. inf2 users must stay on SDK 2.28.
- **vLLM 0.16.0** + vllm-neuron 0.5.0 (up from 0.13.0 / 0.4.1).
- **NKI 0.3.0 (GA)** -- breaking API changes from 0.2.x. ISA kernel behavior may differ.
- **Venv path changed** to `/opt/aws_neuronx_venv_pytorch_inference_vllm_0_16/bin/activate`.
- **multiprocessing.spawn**: vLLM 0.16 spawns EngineCore in a separate process. Scripts MUST have `if __name__ == '__main__':` guard or engine core spawning fails.
- **tie_word_embeddings patch**: Must be applied on disk (not monkey-patched in-process) because spawned process re-imports from disk.
- **Compilation ~3x faster**: 128s on trn2 LNC=2 tp=2 (vs 367s on SDK 2.28 with same config).

### SDK 2.29 Benchmark Results (trn2.3xlarge, LNC=2, tp=2)

Tested 2026-05-01 with NxD Inference 0.9, vLLM 0.16.0, vllm-neuron 0.5.0.

| Metric | Text-Only | Image+Text |
|--------|-----------|------------|
| Throughput (tok/s) | **61.7** | **53.5** |
| Latency (s) | 0.811 | 1.122 |
| Compile time (s) | 128 | 128 |

Comparison with SDK 2.28 (same config: trn2.3xlarge, LNC=2, tp=2, QKV+Attn kernels):

| Metric | SDK 2.28 | SDK 2.29 | Delta |
|--------|----------|----------|-------|
| Text-only (tok/s) | 65.7 | 61.7 | -6.1% |
| Image+text (tok/s) | 61.7 | 53.5 | -13.3% |
| Compile time (s) | 367 | 128 | **-65%** |

**Observations**: Text throughput is comparable (-6%). Image+text shows a larger regression (-13%), likely due to vision encoder changes in NxDI 0.9. Compilation is dramatically faster. For production serving on trn2, either SDK version is viable; SDK 2.29 offers faster iteration with slightly lower throughput.


## Cleanup

If you're done benchmarking, remember to:
1. Save any results you need
2. Stop or terminate your instance to avoid unnecessary charges

```bash
# Check for running Neuron processes
neuron-top
```